# Notebook 10 — Validación del Modelo v2 sobre Set Experimental Final

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  

---

## ¿Qué hace este notebook?

Valida el modelo cognitivo artificial v2 (`best_model_v2.pth`) sobre el set de validación final
de **190 tareas experimentales** con ground truth validado manualmente.

A diferencia de la evaluación del NB05b (que usó pares del conjunto de entrenamiento aumentado),
este notebook evalúa el modelo sobre las **tareas exactas** que también ven los participantes
humanos — mismo diseño factorial, mismas imágenes, mismos bounding boxes A y B.

## Arquitectura del modelo v2

El modelo recibe: imagen completa (6 canales) + coordenadas ROI_A + coordenadas ROI_B.  
Produce: P(A más cercano que B) ∈ [0, 1].  
Predicción: `mas_cercano` si P > 0.5, `mas_lejano` si P ≤ 0.5.

## Ground truth

El ground truth fue validado empíricamente (NB08-09) y corregido para ilusiones visuales.  
Verificación manual: 15/15 imágenes de ilusiones confirmadas correctas.

---

## Celda 1 — Configuración inicial

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

BASE_DIR        = '/content/drive/MyDrive/cognitive-depth-model'
CHECKPOINT_DIR  = os.path.join(BASE_DIR, 'checkpoints')
SPLITS_DIR      = os.path.join(BASE_DIR, 'data/splits')
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')
KITTI_RIGHT_DIR = os.path.join(BASE_DIR, 'data/raw/kitti/data_scene_flow/training/image_3')

CSV_VAL = os.path.join(SPLITS_DIR, 'validation_set_final.csv')
df_val  = pd.read_csv(CSV_VAL)

print(f'\nSet de validación: {len(df_val)} tareas')
print(f'Por dataset:')
print(df_val['dataset'].value_counts().to_string())
print(f'\nBalance ground truth:')
print(df_val['ground_truth'].value_counts().to_string())

## Celda 2 — Definir arquitectura del modelo v2

In [ ]:
# Arquitectura idéntica a NB05b
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch))
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu(x + self.net(x))

class ConvBnSkip(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.main = nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.skip = nn.Sequential(nn.Conv2d(in_ch,out_ch,1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.main(x) + self.skip(x)

class VisualBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.ngl_magno = nn.Sequential(nn.Conv2d(6,8,5,padding=2), nn.BatchNorm2d(8), nn.ReLU(inplace=True), ResBlock(8))
        self.ngl_parvo = nn.Sequential(nn.Conv2d(6,8,3,padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), ResBlock(8))
        self.v1_merge  = nn.Sequential(nn.Conv2d(16,16,1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), ResBlock(16), ResBlock(16))
        self.pool1     = nn.MaxPool2d(2)
        self.v2_thick  = nn.Sequential(ConvBnSkip(16,32), ResBlock(32))
        self.v2_thin   = nn.Sequential(ConvBnSkip(16,32), ResBlock(32))
        self.v2_merge  = nn.Sequential(nn.Conv2d(64,32,1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.pool2     = nn.MaxPool2d(2)
        self.v3_disp   = nn.Sequential(ConvBnSkip(32,64), ResBlock(64), ResBlock(64))
        self.v3_motion = nn.Sequential(ConvBnSkip(32,64), ResBlock(64))
        self.pool3     = nn.MaxPool2d(2)
        self.v4        = nn.Sequential(nn.Conv2d(128,64,1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), ResBlock(64))
        self.v5mt      = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), ResBlock(128), ResBlock(128))
        self.fb_v2_v1  = nn.Sequential(nn.Conv2d(32,16,1), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
    def forward(self, x):
        magno = self.ngl_magno(x); parvo = self.ngl_parvo(x)
        v1    = self.v1_merge(torch.cat([magno, parvo], dim=1))
        v1    = self.pool1(v1)
        thick = self.v2_thick(v1); thin = self.v2_thin(v1)
        v2    = self.v2_merge(torch.cat([thick, thin], dim=1))
        fb    = F.interpolate(self.fb_v2_v1(v2), size=v1.shape[2:], mode='bilinear', align_corners=False)
        v1    = v1 + fb
        thick = self.v2_thick(v1); thin = self.v2_thin(v1)
        v2    = self.v2_merge(torch.cat([thick, thin], dim=1))
        v2    = self.pool2(v2)
        disp  = self.v3_disp(v2); motion = self.v3_motion(v2)
        v3    = self.pool3(torch.cat([disp, motion], dim=1))
        v4    = self.v4(v3)
        return self.v5mt(v4)

class ROIComparator(nn.Module):
    def __init__(self, feat_dim=128, roi_size=4):
        super().__init__()
        self.roi_size = roi_size
        feat_total    = feat_dim * roi_size * roi_size
        self.comparator = nn.Sequential(
            nn.Linear(feat_total*3, 512), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(512, 128),          nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 1),            nn.Sigmoid())
    def roi_pool(self, features, roi):
        B, C, H, W = features.shape
        x1 = roi[:,0]*2-1; y1 = roi[:,1]*2-1
        x2 = roi[:,2]*2-1; y2 = roi[:,3]*2-1
        xs  = torch.stack([x1+(x2-x1)*t for t in torch.linspace(0,1,self.roi_size).to(features.device)], dim=1)
        ys  = torch.stack([y1+(y2-y1)*t for t in torch.linspace(0,1,self.roi_size).to(features.device)], dim=1)
        grid_x = xs.unsqueeze(1).expand(B, self.roi_size, self.roi_size)
        grid_y = ys.unsqueeze(2).expand(B, self.roi_size, self.roi_size)
        grid   = torch.stack([grid_x, grid_y], dim=-1)
        return F.grid_sample(features, grid, align_corners=True, mode='bilinear').flatten(1)
    def forward(self, features, roi_A, roi_B):
        fa = self.roi_pool(features, roi_A)
        fb = self.roi_pool(features, roi_B)
        return self.comparator(torch.cat([fa, fb, fa-fb], dim=1))

class CognitiveDepthModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone   = VisualBackbone()
        self.comparator = ROIComparator(feat_dim=128, roi_size=4)
    def forward(self, img, roi_A, roi_B):
        return self.comparator(self.backbone(img), roi_A, roi_B)

model = CognitiveDepthModelV2().to(device)
ckpt  = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model_v2.pth'),
                   map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f'✓ Modelo v2 cargado')
print(f'  Checkpoint: época {ckpt["epoch"]}, etapa {ckpt["stage"]}')
print(f'  Test loss entrenamiento: {ckpt["test_loss"]:.4f}')

## Celda 3 — Función de inferencia por tarea

In [ ]:
TARGET_SIZE = (128, 384)
mean = np.array([0.485,0.456,0.406,0.485,0.456,0.406], dtype=np.float32)
std  = np.array([0.229,0.224,0.225,0.229,0.224,0.225], dtype=np.float32)

def inferir_tarea(row, model, device, n_repeticiones=3):
    """
    Ejecuta el modelo sobre una tarea del set de validación.
    Retorna el score promedio y la predicción final.
    Se repite n veces para estabilidad (dropout activo = varianza ligera).
    """
    ruta_izq = str(row['ruta_img_izq'])
    if not os.path.exists(ruta_izq):
        return None, 'error'

    img_izq = cv2.imread(ruta_izq)
    if img_izq is None:
        return None, 'error'

    # Imagen derecha: KITTI → image_3, ilusiones → duplicar
    if row['dataset'] == 'KITTI':
        nombre   = os.path.basename(ruta_izq)
        ruta_der = os.path.join(KITTI_RIGHT_DIR, nombre)
        img_der  = cv2.imread(ruta_der) if os.path.exists(ruta_der) else img_izq
        if img_der is None: img_der = img_izq
    else:
        img_der = img_izq.copy()

    h_orig, w_orig = img_izq.shape[:2]
    h_t,    w_t    = TARGET_SIZE

    img_izq = cv2.resize(cv2.cvtColor(img_izq, cv2.COLOR_BGR2RGB), (w_t, h_t))
    img_der = cv2.resize(cv2.cvtColor(img_der, cv2.COLOR_BGR2RGB), (w_t, h_t))

    par = np.concatenate([img_izq, img_der], axis=2).astype(np.float32) / 255.0
    par = (par - mean) / std
    img_tensor = torch.from_numpy(par.transpose(2,0,1)).unsqueeze(0).to(device)

    # Escalar coordenadas ROI al tamaño de inferencia
    img_H = row.get('img_H', h_orig)
    img_W = row.get('img_W', w_orig)
    sx, sy = w_t / img_W, h_t / img_H

    def roi_tensor(x, y, w, h):
        x1 = min(float(x) * sx / w_t, 1.0)
        y1 = min(float(y) * sy / h_t, 1.0)
        x2 = min((float(x) + float(w)) * sx / w_t, 1.0)
        y2 = min((float(y) + float(h)) * sy / h_t, 1.0)
        return torch.tensor([[x1, y1, x2, y2]], dtype=torch.float32).to(device)

    roi_A = roi_tensor(row['A_x'], row['A_y'], row['A_ancho'], row['A_alto'])
    roi_B = roi_tensor(row['B_x'], row['B_y'], row['B_ancho'], row['B_alto'])

    scores = []
    model.eval()
    with torch.no_grad():
        for _ in range(n_repeticiones):
            score = model(img_tensor, roi_A, roi_B).item()
            scores.append(score)

    score_medio = float(np.mean(scores))
    prediccion  = 'mas_cercano' if score_medio > 0.5 else 'mas_lejano'
    return score_medio, prediccion


# Prueba rápida con la primera tarea
row_test = df_val.iloc[0]
score, pred = inferir_tarea(row_test, model, device)
print(f'Prueba de inferencia:')
print(f'  Imagen: {row_test["imagen_id"][:50]}...')
print(f'  Dataset: {row_test["dataset"]}')
print(f'  Score: {score:.4f} | Predicción: {pred}')
print(f'  Ground truth: {row_test["ground_truth"]}')
print(f'  Acierto: {pred == row_test["ground_truth"]}')

## Celda 4 — Validación completa: 190 tareas × 3 repeticiones

In [ ]:
import time

print('Ejecutando validación sobre 190 tareas experimentales...')
print(f'Modelo: best_model_v2.pth (época {ckpt["epoch"]})')
print()

resultados = []
errores    = 0
t0         = time.time()

for rep in range(1, 4):  # 3 repeticiones
    for idx, row in df_val.iterrows():
        score, pred = inferir_tarea(row, model, device, n_repeticiones=1)
        if pred == 'error':
            errores += 1
            continue
        resultados.append({
            'repeticion':            rep,
            'numero_tarea':          row['numero_tarea'],
            'imagen_id':             row['imagen_id'],
            'dataset':               row['dataset'],
            'tipo_tarea':            row['tipo_tarea'],
            'nivel_disparidad':      row['nivel_disparidad'],
            'presencia_distractores':row['presencia_distractores'],
            'ground_truth':          row['ground_truth'],
            'prediccion':            pred,
            'score':                 round(score, 6),
            'acierto':               int(pred == row['ground_truth'])
        })

    acc_rep = np.mean([r['acierto'] for r in resultados if r['repeticion']==rep])
    print(f'  Repetición {rep}: {acc_rep:.4f} ({acc_rep*100:.1f}%)')

elapsed = time.time() - t0
df_resp = pd.DataFrame(resultados)

print()
print(f'✓ Validación completada en {elapsed/60:.1f} minutos')
print(f'  Total respuestas: {len(df_resp)} | Errores: {errores}')
print(f'  Accuracy global:  {df_resp["acierto"].mean():.4f} ({df_resp["acierto"].mean()*100:.1f}%)')

# Guardar respuestas
ruta_resp = os.path.join(RESULTS_DIR, 'model_validation', 'model_responses_v2.csv')
os.makedirs(os.path.dirname(ruta_resp), exist_ok=True)
df_resp.to_csv(ruta_resp, index=False)
print(f'\nRespuestas guardadas: {ruta_resp}')

## Celda 5 — Métricas detalladas por condición factorial

In [ ]:
from sklearn.metrics import f1_score, classification_report

print('='*65)
print('MÉTRICAS DE VALIDACIÓN — MODELO v2')
print('='*65)

# Accuracy global y F1
acc_global = df_resp['acierto'].mean()
f1_global  = f1_score(
    df_resp['ground_truth'].map({'mas_cercano':1,'mas_lejano':0}),
    df_resp['prediccion'].map({'mas_cercano':1,'mas_lejano':0}),
    zero_division=0
)
print(f'Accuracy global: {acc_global:.4f}  ({acc_global*100:.1f}%)')
print(f'F1 global:       {f1_global:.4f}')
print()

# Por repetición
print('Por repetición:')
for rep in [1,2,3]:
    a = df_resp[df_resp['repeticion']==rep]['acierto'].mean()
    print(f'  Rep {rep}: {a:.4f}  ({a*100:.1f}%)')
print()

# Por dataset
print('Por dataset:')
for ds in ['KITTI', '3D_Illusion']:
    sub = df_resp[df_resp['dataset']==ds]
    a   = sub['acierto'].mean()
    print(f'  {ds}: {a:.4f}  ({a*100:.1f}%)')
print()

# Por condición factorial completa
print('Por condición factorial (tipo × disparidad × distractores):')
tabla = df_resp.groupby(
    ['tipo_tarea','nivel_disparidad','presencia_distractores']
)['acierto'].agg(['mean','count']).round(4)
tabla.columns = ['accuracy', 'n_respuestas']
print(tabla.to_string())
print()

# Score medio por condición (confianza del modelo)
print('Score medio (confianza) por dataset:')
print(df_resp.groupby('dataset')['score'].agg(['mean','std']).round(4).to_string())

# Guardar métricas
metricas = {
    'accuracy_global':   acc_global,
    'f1_global':         f1_global,
    'acc_kitti':         df_resp[df_resp['dataset']=='KITTI']['acierto'].mean(),
    'acc_ilusiones':     df_resp[df_resp['dataset']=='3D_Illusion']['acierto'].mean(),
    'acc_rep1':          df_resp[df_resp['repeticion']==1]['acierto'].mean(),
    'acc_rep2':          df_resp[df_resp['repeticion']==2]['acierto'].mean(),
    'acc_rep3':          df_resp[df_resp['repeticion']==3]['acierto'].mean(),
}
pd.DataFrame([metricas]).to_csv(
    os.path.join(RESULTS_DIR, 'metrics', 'model_v2_validation_metrics.csv'), index=False)
print()
print('✓ Métricas guardadas.')

## Celda 6 — Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Validación Modelo v2 — Set Experimental (190 tareas × 3 repeticiones)',
             fontsize=12, fontweight='bold')

# ── 1. Accuracy por condición factorial ──
ax = axes[0]
tabla_plot = df_resp.groupby(['tipo_tarea','nivel_disparidad'])['acierto'].mean().unstack()
tabla_plot.plot(kind='bar', ax=ax, color=['#2980b9','#e74c3c'], alpha=0.85, edgecolor='white')
ax.set_title('Accuracy por tipo × disparidad', fontsize=10)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.7, label='Azar')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=8)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

# ── 2. Distribución de scores ──
ax = axes[1]
for ds, color, label in [('KITTI','#2980b9','KITTI'), ('3D_Illusion','#e74c3c','Ilusiones')]:
    sub = df_resp[df_resp['dataset']==ds]
    ax.hist(sub['score'], bins=30, alpha=0.6, color=color, label=label, density=True)
ax.axvline(0.5, color='black', ls='--', lw=1.5, label='Umbral (0.5)')
ax.set_title('Distribución de scores del modelo', fontsize=10)
ax.set_xlabel('P(A más cercano)')
ax.set_ylabel('Densidad')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 3. Accuracy por repetición ──
ax = axes[2]
reps = [1, 2, 3]
accs = [df_resp[df_resp['repeticion']==r]['acierto'].mean() for r in reps]
ax.bar([f'Rep {r}' for r in reps], [a*100 for a in accs],
       color=['#27ae60','#27ae60','#27ae60'], alpha=0.85, edgecolor='white')
ax.set_ylim(0, 110)
ax.set_title('Accuracy por repetición', fontsize=10)
ax.set_ylabel('Accuracy (%)')
for i, a in enumerate(accs):
    ax.text(i, a*100+1, f'{a*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.axhline(50, color='gray', ls='--', lw=1, alpha=0.7)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
ruta_fig = os.path.join(RESULTS_DIR, 'visualizations', 'model_v2_validation_results.png')
os.makedirs(os.path.dirname(ruta_fig), exist_ok=True)
plt.savefig(ruta_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {ruta_fig}')

## Celda 7 — Análisis de errores y ejemplos representativos

In [ ]:
# Análisis de consistencia entre repeticiones
pivot = df_resp.pivot_table(
    index='imagen_id', columns='repeticion', values='acierto'
)
pivot.columns = ['rep1','rep2','rep3']
pivot['consistente'] = (pivot['rep1']==pivot['rep2']) & (pivot['rep2']==pivot['rep3'])
pivot['aciertos']    = pivot[['rep1','rep2','rep3']].sum(axis=1)

print('Consistencia entre repeticiones:')
print(f'  Imágenes con 3/3 aciertos:  {(pivot["aciertos"]==3).sum()} ({(pivot["aciertos"]==3).mean()*100:.1f}%)')
print(f'  Imágenes con 2/3 aciertos:  {(pivot["aciertos"]==2).sum()} ({(pivot["aciertos"]==2).mean()*100:.1f}%)')
print(f'  Imágenes con 1/3 aciertos:  {(pivot["aciertos"]==1).sum()} ({(pivot["aciertos"]==1).mean()*100:.1f}%)')
print(f'  Imágenes con 0/3 aciertos:  {(pivot["aciertos"]==0).sum()} ({(pivot["aciertos"]==0).mean()*100:.1f}%)')
print()

# Tareas más difíciles para el modelo (0 aciertos en las 3 repeticiones)
ids_dificiles = pivot[pivot['aciertos']==0].index.tolist()
if ids_dificiles:
    df_dificiles = df_val[df_val['imagen_id'].isin(ids_dificiles)]
    print(f'Tareas con 0/3 aciertos ({len(ids_dificiles)} imágenes):')
    print(df_dificiles[['imagen_id','dataset','tipo_tarea',
                         'nivel_disparidad','presencia_distractores',
                         'ground_truth']].to_string(index=False))
else:
    print('No hay tareas con 0/3 aciertos — el modelo fue consistente en todas las imágenes.')

# Resumen final
print()
print('='*65)
print('RESUMEN FINAL — MODELO v2 EN TAREA EXPERIMENTAL')
print('='*65)
print(f'  Accuracy global:    {df_resp["acierto"].mean():.4f}  ({df_resp["acierto"].mean()*100:.1f}%)')
print(f'  KITTI:              {df_resp[df_resp["dataset"]=="KITTI"]["acierto"].mean():.4f}  ({df_resp[df_resp["dataset"]=="KITTI"]["acierto"].mean()*100:.1f}%)')
print(f'  Ilusiones:          {df_resp[df_resp["dataset"]=="3D_Illusion"]["acierto"].mean():.4f}  ({df_resp[df_resp["dataset"]=="3D_Illusion"]["acierto"].mean()*100:.1f}%)')
print(f'  Imágenes perfectas: {(pivot["aciertos"]==3).sum()}/190')
print()
print('Próximo paso → Recolección de datos con participantes humanos (NB11 + App Quest 2)')
print('               y comparación modelo vs. humanos (NB12)')